In [ ]:
from google.colab import drive
import pickle
import pandas as pd
from collections import defaultdict
import random
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
import timm
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install rouge_score

In [ ]:
!pip install pycocoevalcap

In [ ]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

import pandas as pd
import torch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
# Copying 8k to Colabs local SSD
if not os.path.exists("/content/flickr8k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr8k.zip" /content/

In [ ]:
if not os.path.isdir("/content/flickr8k/Images"):
  !unzip "/content/flickr8k.zip" -d "/content/flickr8k"

In [ ]:
print("Images:", len(os.listdir("/content/flickr8k/Images")))

Images: 8091


In [ ]:
# Copying 30k to Colabs local SSD
if not os.path.exists("/content/flickr30k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr30k.zip" /content/

In [ ]:
if not os.path.isdir("/content/flickr30k/Images"):
  !unzip "/content/flickr30k.zip" -d "/content/flickr30k"

In [ ]:
print("Images:", len(os.listdir("/content/flickr30k/Images")))

Images: 31811


In [ ]:
DATASETS = {
    "flickr8k": {
        "ROOT": "/content/flickr8k",
        "IMAGE_DIR": "/content/flickr8k/Images",
        "CAPTION_FILE": "/content/flickr8k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/flickr8k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/vocab.pkl"
    },
    "flickr30k": {
        "ROOT": "/content/flickr30k",
        "IMAGE_DIR": "/content/flickr30k/Images",
        "CAPTION_FILE": "/content/flickr30k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/flickr30k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/vocab.pkl"
    }
}

In [ ]:
def image_caption_map(train_df,val_df,test_df):
  train_caption_map = defaultdict(list)
  val_caption_map = defaultdict(list)
  test_caption_map = defaultdict(list)

  bad_img = "861608773_bdafd5c996.jpg"

  train_caption_map.pop(bad_img, None)
  val_caption_map.pop(bad_img, None)
  test_caption_map.pop(bad_img, None)

  for _, row in train_df.iterrows():
      train_caption_map[row["image"]].append(row["caption"])

  for _, row in val_df.iterrows():
      val_caption_map[row["image"]].append(row["caption"])

  for _, row in test_df.iterrows():
      test_caption_map[row["image"]].append(row["caption"])
  return train_caption_map,val_caption_map,test_caption_map



In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Encoding

def encode_caption(text, vocab):

    tokens = text.split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [ ]:
class FlickrCaptionDataset(Dataset):

    def __init__(
            self,
            caption_map,
            image_dir,
            vocab,
            transform=None,
            random_caption=True):

        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.random_caption = random_caption

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        captions = self.caption_map[image_name]

        if self.random_caption:
            caption = random.choice(captions)
        else:
            caption = captions[0]

        try:
            image = Image.open(
                os.path.join(self.image_dir, image_name)
            ).convert("RGB")
        except Exception:
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            image = self.transform(image)

        caption = encode_caption(caption, self.vocab)

        input_caption = torch.tensor(
            caption[:-1],
            dtype=torch.long
        )

        target_caption = torch.tensor(
            caption[1:],
            dtype=torch.long
        )

        return image, input_caption, target_caption

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class FlickrCaptionTestDataset(Dataset):

    def __init__(
            self,
            caption_map,
            image_dir,
            transform=None
    ):

        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        # List containing all 5 captions
        captions = self.caption_map[image_name]

        try:
            image = Image.open(
                os.path.join(self.image_dir, image_name)
            ).convert("RGB")

        except Exception:
            return self.__getitem__(
                (idx + 1) % len(self)
            )

        if self.transform:
            image = self.transform(image)

        return image, captions

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []
    input_captions = []
    target_captions_list = []

    for image, input_caption, target_caption in batch:
        images.append(image)
        input_captions.append(input_caption)
        target_captions_list.append(target_caption)

    images = torch.stack(images)

    input_captions = pad_sequence(
        input_captions,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    padded_target_captions = pad_sequence(
        target_captions_list,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    return images, input_captions, padded_target_captions

In [ ]:
#encoder
import torchvision.models as tv_models
import torch.nn as nn

class ViTImageEncoder(nn.Module):

    def __init__(self, embed_dim=512):
        super().__init__()

        vit = tv_models.vit_b_16(weights=tv_models.ViT_B_16_Weights.IMAGENET1K_V1)

        # torchvision's ViT internals
        self.conv_proj = vit.conv_proj
        self.class_token = vit.class_token
        self.encoder = vit.encoder


        # freeze everything by default
        for p in self.parameters():
            p.requires_grad = False

        # unfreeze last 4 transformer blocks
        for block in list(self.encoder.layers.children())[-4:]:
            for p in block.parameters():
                p.requires_grad = True

        self.projection = nn.Sequential(
            nn.Linear(768,768),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(768,embed_dim),
            nn.LayerNorm(embed_dim)
        )

    def _process_input(self, x):
        n, c, h, w = x.shape
        p = 16  # patch size
        x = self.conv_proj(x)
        x = x.reshape(n, 768, (h // p) * (w // p))
        x = x.permute(0, 2, 1)
        return x

    def forward(self, x):
        n = x.shape[0]
        x = self._process_input(x)

        batch_class_token = self.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)

        x = self.encoder(x)

        # CLS token
        x = x[:, 0]

        x = self.projection(x)
        return x

In [ ]:
class LSTMCaptionDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        hidden_dim=512,
        pad_idx=0
    ):
        super().__init__()

        # Word embedding
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            dropout=0.0
        )
        for name, param in self.lstm.named_parameters():
          if "weight_ih" in name:
              nn.init.xavier_uniform_(param)
          elif "weight_hh" in name:
              nn.init.orthogonal_(param)
          elif "bias" in name:
              nn.init.zeros_(param)
        self.norm = nn.LayerNorm(hidden_dim)

        self.dropout = nn.Dropout(0.2)

        # Output vocabulary prediction
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, vocab_size)
        )

        self.image_proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.LayerNorm(embed_dim)
        )
        nn.init.xavier_uniform_(self.embedding.weight)

        # Initialize all Linear layers
        for m in self.modules():
          if isinstance(m, nn.Linear):
              nn.init.xavier_uniform_(m.weight)
              if m.bias is not None:
                  nn.init.zeros_(m.bias)
        self.fc[-1].weight = self.embedding.weight
        with torch.no_grad():
          self.embedding.weight[pad_idx].zero_()

    def forward(self, image_features, captions):

      embeddings = self.embedding(captions)

      image_features = self.image_proj(image_features)
      image_features = image_features.unsqueeze(1)

      inputs = torch.cat(
          (image_features, embeddings),
          dim=1
      )

      outputs, _ = self.lstm(inputs)

      outputs = outputs[:, 1:, :]      # remove image timestep
      outputs = self.norm(outputs)
      outputs = self.dropout(outputs)

      outputs = self.fc(outputs)

      return outputs

In [ ]:
#Joint Model
class ViTLSTMCaptioningModel(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):

        # Image embedding from ViT
        image_features = self.encoder(images)

        # Caption decoder
        outputs = self.decoder(
            image_features,
            captions
        )

        return outputs

    def generate_caption(self,images,vocab,max_length=30,beam_size=3):
        self.eval()
        with torch.inference_mode():
            device_type = images.device.type
            with torch.autocast(device_type=device_type,enabled=(device_type == "cuda")):
                # Beam search currently supports one image
                images = images[:1]
                image_features = self.encoder(images)
                image_features = self.decoder.image_proj(image_features)
                _, hidden = self.decoder.lstm(image_features.unsqueeze(1))
                beams = [([vocab["<SOS>"]],hidden,0.0)]
                completed = []
                for step in range(max_length):
                    candidates = []
                    for tokens, hidden_state, score in beams:
                        # Already finished
                        if tokens[-1] == vocab["<EOS>"]:
                            completed.append((tokens, hidden_state, score))
                            continue
                        last_token = torch.tensor([tokens[-1]],device=images.device)
                        inputs = self.decoder.embedding(last_token).unsqueeze(1)

                        outputs, new_hidden = self.decoder.lstm(
                            inputs,
                            hidden_state
                        )

                        outputs = self.decoder.norm(outputs)
                        outputs = self.decoder.dropout(outputs)

                        scores = self.decoder.fc(outputs.squeeze(1))

                        # Prevent special tokens
                        scores[:, vocab["<PAD>"]] = float("-inf")
                        scores[:, vocab["<SOS>"]] = float("-inf")
                        scores[:, vocab["<UNK>"]] = float("-inf")

                        if step < 3:
                            scores[:, vocab["<EOS>"]] = float("-inf")

                        log_probs = F.log_softmax(
                            scores,
                            dim=-1
                        )
                        values, indices = torch.topk(
                            log_probs,
                            beam_size*3,
                            dim=-1
                        )

                        for k in range(beam_size * 3):
                            token = indices[0, k].item()

                            # repetition blocking
                            # prevent immediate repetition
                            if len(tokens) > 0 and token == tokens[-1]:
                                continue
                            if token in tokens[-3:]:
                              continue
                            # Trigram blocking
                            if len(tokens) >= 3:
                                trigram = (tokens[-2], tokens[-1], token)
                                exists = False
                                for i in range(len(tokens)-2):
                                    if (tokens[i],tokens[i+1],tokens[i+2]) == trigram:
                                        exists = True
                                        break
                                if exists:
                                    continue
                            new_tokens = tokens + [token]
                            new_score = score + values[0, k].item()
                            new_hidden_clone = (new_hidden[0].clone(),new_hidden[1].clone())

                            candidates.append((new_tokens,new_hidden_clone,new_score))
                    if len(candidates) == 0:
                        break
                    # Length normalization
                    alpha = 0.7

                    candidates = sorted(
                        candidates,
                        key=lambda x:
                        x[2] / (((5 + len(x[0])) / 6) ** alpha),
                        reverse=True
                    )
                    beams = candidates[:beam_size]
                    if all(beam[0][-1] == vocab["<EOS>"] for beam in beams):
                        completed.extend(beams)
                        break
                if len(completed) == 0:
                    completed = beams

                completed = sorted(
                    completed,
                    key=lambda x:
                    x[2] / (((5 + len(x[0])) / 6) ** alpha),
                    reverse=True
                )
                best = completed[0][0]
                # Remove SOS
                best = best[1:]
                # Trim after EOS
                if vocab["<EOS>"] in best:
                    best = best[:best.index(vocab["<EOS>"])]
                return [best]

In [ ]:
def build_image_text_decoder(vocab,device):
  encoder = ViTImageEncoder(embed_dim=512)

  decoder = LSTMCaptionDecoder(
      vocab_size=len(vocab),
      embed_dim=512,
      hidden_dim=512,
      pad_idx=vocab["<PAD>"]
  )

  model = ViTLSTMCaptioningModel(
      encoder,
      decoder
  ).to(device)

  return model

In [ ]:
def define_optimizer(model):

    optimizer = torch.optim.AdamW(
    [
        {
            "params": filter(
                lambda p: p.requires_grad,
                model.encoder.encoder.parameters()
            ),
            "lr":5e-6
        },
        {
            "params": model.encoder.projection.parameters(),
            "lr":2e-4
        },
        {
            "params": model.decoder.parameters(),
            "lr":2e-4
        }
    ],
    betas=(0.9,0.98),
    weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    return optimizer, scheduler

In [ ]:
def evaluate(model, dataloader, criterion, device):

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for images, input_captions, target_captions in dataloader:

            images = images.to(device)
            input_captions = input_captions.to(device)
            target_captions = target_captions.to(device)

            outputs = model(
                images,
                input_captions
            )

            loss = criterion(
                outputs.reshape(-1, outputs.size(-1)),
                target_captions.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import os
import time
import torch
import torch.nn as nn

def model_training(
        model,
        train_loader,
        val_loader,
        optimizer,
        scheduler,
        criterion,
        dataset_name):

    NUM_EPOCHS = 50

    best_val_loss = float("inf")

    patience = 8
    epochs_without_improvement = 0

    SAVE_DIR = f"/content/drive/MyDrive/MMCaptioning/C3/{dataset_name}"
    os.makedirs(SAVE_DIR, exist_ok=True)

    BEST_MODEL_PATH = os.path.join(
        SAVE_DIR,
        "best_captioning_C3_model.pth"
    )

    for epoch in range(NUM_EPOCHS):

        print(f"\n================ Epoch {epoch+1}/{NUM_EPOCHS} ================")

        model.train()

        running_loss = 0.0

        epoch_start = time.time()

        for batch_idx, (images, input_captions, target_captions) in enumerate(train_loader):

            batch_start = time.time()

            images = images.to(device, non_blocking=True)
            input_captions = input_captions.to(device, non_blocking=True)
            target_captions = target_captions.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(
                images,
                input_captions
            )

            loss = criterion(
                outputs.reshape(-1, outputs.size(-1)),
                target_captions.reshape(-1)
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            running_loss += loss.item()

            print(
                f"Batch {batch_idx+1:03d}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Time: {time.time()-batch_start:.2f}s"
            )

        train_loss = running_loss / len(train_loader)

        train_time = time.time() - epoch_start

        # Validation

        val_start = time.time()

        val_loss = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        scheduler.step()

        val_time = time.time() - val_start

        print("\n---------------- Summary ----------------")

        print(f"Train Loss      : {train_loss:.4f}")
        print(f"Validation Loss : {val_loss:.4f}")
        print(f"Training Time   : {train_time:.2f}s")
        print(f"Validation Time : {val_time:.2f}s")

        print(
            f"GPU Memory Used : "
            f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
        )

        # Save Best

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "epoch": epoch,
                    "val_loss": val_loss
                },
                BEST_MODEL_PATH
            )

            print(f"✓ Best model saved (Val Loss: {val_loss:.4f})")

        else:

            epochs_without_improvement += 1

            print(
                f"No improvement for "
                f"{epochs_without_improvement}/{patience} epochs"
            )

        if epochs_without_improvement >= patience:

            print("\nEarly stopping triggered!")

            break

    checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    return model, BEST_MODEL_PATH

In [ ]:
import gc
def reset_resources():
  gc.collect()
  torch.cuda.empty_cache()
  print("\nMemory after cleanup")
  print("Allocated:",torch.cuda.memory_allocated()/1024**3)
  print("Reserved:",torch.cuda.memory_reserved()/1024**3)

In [ ]:
def caption_test_collate_fn(batch):

    images = []
    all_references = []

    for image, references in batch:
        images.append(image)
        all_references.append(references)

    images = torch.stack(images)

    return images, all_references

In [ ]:
def prepare_caption_dataloaders(
        train_caption_map,
        val_caption_map,
        test_caption_map):

    BATCH_SIZE = 192

    train_dataset = FlickrCaptionDataset(
        train_caption_map,
        IMAGE_DIR,
        vocab,
        train_transform,
        random_caption=True
    )

    val_dataset = FlickrCaptionDataset(
        val_caption_map,
        IMAGE_DIR,
        vocab,
        image_transform,
        random_caption=False
    )

    # Different dataset for evaluation
    test_dataset = FlickrCaptionTestDataset(
        test_caption_map,
        IMAGE_DIR,
        image_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=8,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=8,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=caption_test_collate_fn,   # Different collate function
        num_workers=8,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader

In [ ]:
def model_testing(model,checkpoint_path,test_loader,vocab,device):
  checkpoint = torch.load(checkpoint_path,map_location=device)
  model.load_state_dict(checkpoint["model_state_dict"])
  model.to(device)
  model.eval()



  idx2word = {v: k for k, v in vocab.items()}

  references = []
  predictions = []

  rouge = rouge_scorer.RougeScorer(["rougeL"],use_stemmer=True)

  rouge_scores = []
  meteor_scores = []

  coco_refs = {}
  coco_preds = {}

  sample_id = 0

  print("Generating captions...")

  with torch.inference_mode():

      for images, all_references in test_loader:

        images = images.to(device)   # move BEFORE running encoder

        predicted_ids = []

        for i in range(images.size(0)):
            pred = model.generate_caption(
                images[i:i+1],
                vocab
            )
            predicted_ids.extend(pred)

        for i in range(images.size(0)):
            # Prediction
            pred_words = []
            for idx in predicted_ids[i]:
              idx = int(idx)

              if idx == vocab["<EOS>"]:
                  break

              if idx in (
                  vocab["<PAD>"],
                  vocab["<SOS>"]
              ):
                  continue

              pred_words.append(idx2word[idx])

            pred_sentence = " ".join(pred_words)

            # References (5 captions)
            ref_words = [ref.split() for ref in all_references[i]]
            references.append(ref_words)
            predictions.append(pred_words)

            # ROUGE-L
            rouge_values = []

            for ref in all_references[i]:

                rouge_values.append(rouge.score(ref,pred_sentence)["rougeL"].fmeasure)

            rouge_scores.append(max(rouge_values))

            # METEOR
            meteor_scores.append(meteor_score(ref_words,pred_words))

            # COCO format
            coco_refs[sample_id] = [str(ref) for ref in all_references[i]]

            coco_preds[sample_id] = [str(pred_sentence)]

            sample_id += 1

  # BLEU
  smoothie = SmoothingFunction().method4
  bleu1 = corpus_bleu(references,predictions,weights=(1,0,0,0),smoothing_function=smoothie)

  bleu2 = corpus_bleu(references,predictions,weights=(0.5,0.5,0,0),smoothing_function=smoothie)

  bleu3 = corpus_bleu(references,predictions,weights=(0.33,0.33,0.33,0),smoothing_function=smoothie)

  bleu4 = corpus_bleu(references,predictions,weights=(0.25,0.25,0.25,0.25),smoothing_function=smoothie)

  # CIDEr
  cider_scorer = Cider()

  cider_score, _ = cider_scorer.compute_score(coco_refs,coco_preds)

  # SPICE

  # Results

  results = pd.DataFrame({

      "Metric":[
          "BLEU-1",
          "BLEU-2",
          "BLEU-3",
          "BLEU-4",
          "ROUGE-L",
          "METEOR",
          "CIDEr",

      ],

      "Score":[
          bleu1,
          bleu2,
          bleu3,
          bleu4,
          sum(rouge_scores)/len(rouge_scores),
          sum(meteor_scores)/len(meteor_scores),
          cider_score,

      ]

  })
  print(coco_refs[0])
  print(coco_preds[0])
  display(results)

  return results

In [ ]:
for dataset_name, cfg in DATASETS.items():

    torch.cuda.reset_peak_memory_stats()

    print("\n************************")
    print(f"\nProcessing - {dataset_name}")
    print("\n************************\n")

    ROOT = cfg["ROOT"]
    IMAGE_DIR = cfg["IMAGE_DIR"]
    CAPTION_FILE = cfg["CAPTION_FILE"]

    with open(cfg["flickr_split"], "rb") as f:
        split = pickle.load(f)

    with open(cfg["vocab"], "rb") as f:
        vocab = pickle.load(f)


    #Build Model

    model = build_image_text_decoder(
        vocab,
        device
    )

    optimizer, scheduler = define_optimizer(model)

    # Dataset

    df = pd.read_csv(CAPTION_FILE)

    train_imgs = split["train"]
    val_imgs = split["val"]
    test_imgs = split["test"]

    print(
        f"Train: {len(train_imgs)} | "
        f"Val: {len(val_imgs)} | "
        f"Test: {len(test_imgs)}"
    )

    train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
    val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
    test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

    train_df = train_df.dropna(subset=["caption"]).reset_index(drop=True)
    val_df = val_df.dropna(subset=["caption"]).reset_index(drop=True)
    test_df = test_df.dropna(subset=["caption"]).reset_index(drop=True)

    train_caption_map, val_caption_map, test_caption_map = image_caption_map(
        train_df,
        val_df,
        test_df
    )

    train_loader, val_loader, test_loader = prepare_caption_dataloaders(
        train_caption_map,
        val_caption_map,
        test_caption_map
    )
    criterion = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"],label_smoothing=0.1)

    #Train

    best_model, FINAL_MODEL_PATH = model_training(
        model,
        train_loader,
        val_loader,
        optimizer,
        scheduler,
        criterion,
        dataset_name
    )

    #Test

    model_testing(
        best_model,
        FINAL_MODEL_PATH,
        test_loader,
        vocab,
        device
    )

    # Cleanup

    del model
    del optimizer
    del scheduler

    del train_loader
    del val_loader
    del test_loader

    del train_df
    del val_df
    del test_df

    del train_caption_map
    del val_caption_map
    del test_caption_map

    del split
    del vocab
    del df

    reset_resources()



************************

Processing - flickr8k

************************

Train: 6068 | Val: 1011 | Test: 1012

================ Epoch 1/50 ================
Batch 001/32 | Loss: 7.9909 | Time: 0.74s
Batch 002/32 | Loss: 7.8180 | Time: 0.71s
Batch 003/32 | Loss: 7.6380 | Time: 0.71s
Batch 004/32 | Loss: 7.4562 | Time: 0.72s
Batch 005/32 | Loss: 7.2704 | Time: 0.71s
Batch 006/32 | Loss: 7.0728 | Time: 0.71s
Batch 007/32 | Loss: 6.8629 | Time: 0.71s
Batch 008/32 | Loss: 6.6541 | Time: 0.71s
Batch 009/32 | Loss: 6.4941 | Time: 0.72s
Batch 010/32 | Loss: 6.3140 | Time: 0.71s
Batch 011/32 | Loss: 6.0902 | Time: 0.71s
Batch 012/32 | Loss: 5.9091 | Time: 0.71s
Batch 013/32 | Loss: 5.9343 | Time: 0.71s
Batch 014/32 | Loss: 5.7025 | Time: 0.71s
Batch 015/32 | Loss: 5.6328 | Time: 0.71s
Batch 016/32 | Loss: 5.5884 | Time: 0.71s
Batch 017/32 | Loss: 5.5406 | Time: 0.71s
Batch 018/32 | Loss: 5.4091 | Time: 0.71s
Batch 019/32 | Loss: 5.3895 | Time: 0.71s
Batch 020/32 | Loss: 5.3943 | Time: 0.71s
B

,Metric,Score
0,BLEU-1,0.516005
1,BLEU-2,0.332974
2,BLEU-3,0.217237
3,BLEU-4,0.137155
4,ROUGE-L,0.429314
5,METEOR,0.374411
6,CIDEr,0.378846


Streaming output truncated to the last 5000 lines.
GPU Memory Used : 2.44 GB
✓ Best model saved (Val Loss: 3.9421)

================ Epoch 14/50 ================
Batch 001/125 | Loss: 3.7852 | Time: 0.74s
Batch 002/125 | Loss: 3.6958 | Time: 0.72s
Batch 003/125 | Loss: 3.8142 | Time: 0.72s
Batch 004/125 | Loss: 3.7024 | Time: 0.72s
Batch 005/125 | Loss: 3.7031 | Time: 0.72s
Batch 006/125 | Loss: 3.8177 | Time: 0.72s
Batch 007/125 | Loss: 3.7551 | Time: 0.72s
Batch 008/125 | Loss: 3.7157 | Time: 0.72s
Batch 009/125 | Loss: 3.8267 | Time: 0.72s
Batch 010/125 | Loss: 3.8585 | Time: 0.72s
Batch 011/125 | Loss: 3.8582 | Time: 0.72s
Batch 012/125 | Loss: 3.7588 | Time: 0.72s
Batch 013/125 | Loss: 3.7918 | Time: 0.72s
Batch 014/125 | Loss: 3.7407 | Time: 0.71s
Batch 015/125 | Loss: 3.8496 | Time: 0.72s
Batch 016/125 | Loss: 3.8357 | Time: 0.72s
Batch 017/125 | Loss: 3.7875 | Time: 0.72s
Batch 018/125 | Loss: 3.7651 | Time: 0.72s
Batch 019/125 | Loss: 3.7693 | Time: 0.72s
Batch 020/125 | Loss:

,Metric,Score
0,BLEU-1,0.487510
1,BLEU-2,0.307515
2,BLEU-3,0.194641
3,BLEU-4,0.118764
4,ROUGE-L,0.389961
5,METEOR,0.354118
6,CIDEr,0.276262



Memory after cleanup
Allocated: 1.6676154136657715
Reserved: 4.154296875
